# Bayesian Inference - Exercises

Practice problems for Bayesian concepts.

## Exercises
1. Beta-Binomial: A/B Testing
2. Normal Conjugate Prior
3. Prior Sensitivity Analysis
4. Posterior Predictive

In [ ]:
import numpy as np
from scipy import stats
from scipy.special import beta as beta_func, comb
import matplotlib.pyplot as plt

np.random.seed(42)

---
## Exercise 1: Beta-Binomial A/B Testing

**Scenario**: You're running an A/B test on two model versions.

- Model A: 45 correct predictions out of 60
- Model B: 52 correct predictions out of 70

**Tasks**:
1. Use Beta(1,1) uniform prior for both
2. Calculate posterior distributions
3. What is P(θ_B > θ_A)? (probability B is better)
4. Visualize both posteriors

In [ ]:
# Data
correct_A, total_A = 45, 60
correct_B, total_B = 52, 70

# Prior
alpha_prior, beta_prior = 1, 1

# Your solution here

### Solution

In [ ]:
# Data
correct_A, total_A = 45, 60
correct_B, total_B = 52, 70

# Prior
alpha_prior, beta_prior = 1, 1

print("BAYESIAN A/B TESTING")
print("="*55)

print(f"\n📊 Data:")
print(f"  Model A: {correct_A}/{total_A} = {correct_A/total_A:.2%}")
print(f"  Model B: {correct_B}/{total_B} = {correct_B/total_B:.2%}")

# 1 & 2. Posterior distributions
alpha_A = alpha_prior + correct_A
beta_A = beta_prior + total_A - correct_A

alpha_B = alpha_prior + correct_B
beta_B = beta_prior + total_B - correct_B

print(f"\n📊 Posteriors:")
print(f"  Model A: Beta({alpha_A}, {beta_A})")
print(f"    Mean: {alpha_A/(alpha_A+beta_A):.4f}")
print(f"    95% CI: {stats.beta.ppf([0.025, 0.975], alpha_A, beta_A)}")

print(f"  Model B: Beta({alpha_B}, {beta_B})")
print(f"    Mean: {alpha_B/(alpha_B+beta_B):.4f}")
print(f"    95% CI: {stats.beta.ppf([0.025, 0.975], alpha_B, beta_B)}")

# 3. P(θ_B > θ_A) via Monte Carlo
n_samples = 100000
samples_A = np.random.beta(alpha_A, beta_A, n_samples)
samples_B = np.random.beta(alpha_B, beta_B, n_samples)
prob_B_better = np.mean(samples_B > samples_A)

print(f"\n3️⃣ P(θ_B > θ_A) = {prob_B_better:.4f} ({prob_B_better:.1%})")

# Also calculate expected lift
expected_lift = np.mean(samples_B - samples_A)
lift_ci = np.percentile(samples_B - samples_A, [2.5, 97.5])

print(f"")
print(f"  Expected lift (B - A): {expected_lift:.4f} ({expected_lift*100:.2f}%)")
print(f"  95% CI on lift: [{lift_ci[0]:.4f}, {lift_ci[1]:.4f}]")

# 4. Visualization
theta = np.linspace(0.5, 1.0, 200)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Posterior comparison
axes[0].plot(theta, stats.beta.pdf(theta, alpha_A, beta_A), 'b-', lw=2, label=f'Model A: {correct_A/total_A:.1%}')
axes[0].plot(theta, stats.beta.pdf(theta, alpha_B, beta_B), 'r-', lw=2, label=f'Model B: {correct_B/total_B:.1%}')
axes[0].fill_between(theta, stats.beta.pdf(theta, alpha_A, beta_A), alpha=0.3)
axes[0].fill_between(theta, stats.beta.pdf(theta, alpha_B, beta_B), alpha=0.3)
axes[0].set_xlabel('θ (accuracy)')
axes[0].set_ylabel('Density')
axes[0].set_title('Posterior Distributions')
axes[0].legend()

# Lift distribution
lift_samples = samples_B - samples_A
axes[1].hist(lift_samples, bins=100, density=True, alpha=0.7, color='green')
axes[1].axvline(0, color='black', linestyle='--', lw=2, label='No difference')
axes[1].axvline(expected_lift, color='red', linestyle='-', lw=2, label=f'E[lift]={expected_lift:.3f}')
axes[1].set_xlabel('Lift (θ_B - θ_A)')
axes[1].set_ylabel('Density')
axes[1].set_title(f'Lift Distribution: P(B>A) = {prob_B_better:.1%}')
axes[1].legend()

plt.tight_layout()
plt.show()

# Decision
print("\n🎯 Decision:")
if prob_B_better > 0.95:
    print(f"  Strong evidence B is better ({prob_B_better:.1%} confidence)")
elif prob_B_better > 0.80:
    print(f"  Moderate evidence B is better ({prob_B_better:.1%} confidence)")
    print("  Consider collecting more data")
else:
    print(f"  Inconclusive ({prob_B_better:.1%} confidence)")
    print("  Need more data to decide")

---
## Exercise 2: Normal Conjugate Prior

**Scenario**: Estimating average inference time of a model.

- Prior belief: μ ~ N(50, 10²) (expect ~50ms, uncertain)
- Data: 8 measurements with mean 42ms, known σ = 5ms

**Tasks**:
1. Calculate posterior distribution
2. Find 95% credible interval
3. Compare posterior mean with prior mean and sample mean

In [ ]:
# Prior
mu_prior = 50
sigma_prior = 10

# Data
x_bar = 42
n = 8
sigma_data = 5  # Known population std

# Your solution here

### Solution

In [ ]:
# Prior
mu_prior = 50
sigma_prior = 10

# Data
x_bar = 42
n = 8
sigma_data = 5  # Known population std

print("NORMAL CONJUGATE PRIOR")
print("="*55)

print(f"\n📊 Prior: N({mu_prior}, {sigma_prior}²)")
print(f"📊 Data: x̄ = {x_bar}ms, n = {n}, σ = {sigma_data}ms")

# 1. Calculate posterior
precision_prior = 1 / sigma_prior**2
precision_data = n / sigma_data**2
precision_post = precision_prior + precision_data

mu_post = (precision_prior * mu_prior + precision_data * x_bar) / precision_post
sigma_post = np.sqrt(1 / precision_post)

print(f"\n1️⃣ Posterior Calculation:")
print(f"  Prior precision: 1/{sigma_prior}² = {precision_prior:.4f}")
print(f"  Data precision: {n}/{sigma_data}² = {precision_data:.4f}")
print(f"  Posterior precision: {precision_post:.4f}")
print(f"")
print(f"  Posterior: N({mu_post:.4f}, {sigma_post:.4f}²)")

# 2. 95% Credible Interval
ci = stats.norm.ppf([0.025, 0.975], mu_post, sigma_post)

print(f"\n2️⃣ 95% Credible Interval: [{ci[0]:.2f}, {ci[1]:.2f}] ms")

# 3. Compare estimates
print(f"\n3️⃣ Comparison of Estimates:")
print(f"  Prior mean:     {mu_prior:.2f} ms")
print(f"  Sample mean:    {x_bar:.2f} ms")
print(f"  Posterior mean: {mu_post:.2f} ms")

weight_prior = precision_prior / precision_post
weight_data = precision_data / precision_post
print(f"")
print(f"  Weights: Prior = {weight_prior:.2%}, Data = {weight_data:.2%}")
print(f"  Posterior = {weight_prior:.2f}×{mu_prior} + {weight_data:.2f}×{x_bar} = {mu_post:.2f}")

# Visualization
x = np.linspace(25, 65, 200)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(x, stats.norm.pdf(x, mu_prior, sigma_prior), 'b--', lw=2, 
        label=f'Prior N({mu_prior}, {sigma_prior}²)')
ax.plot(x, stats.norm.pdf(x, x_bar, sigma_data/np.sqrt(n)), 'g:', lw=2, 
        label=f'Likelihood (sampling dist)')
ax.plot(x, stats.norm.pdf(x, mu_post, sigma_post), 'r-', lw=3, 
        label=f'Posterior N({mu_post:.1f}, {sigma_post:.1f}²)')

ax.axvline(mu_prior, color='b', linestyle='--', alpha=0.3)
ax.axvline(x_bar, color='g', linestyle=':', alpha=0.3)
ax.axvline(mu_post, color='r', linestyle='-', alpha=0.3)

ax.axvspan(ci[0], ci[1], alpha=0.2, color='red', label='95% CI')

ax.set_xlabel('Inference Time (ms)')
ax.set_ylabel('Density')
ax.set_title('Bayesian Estimation of Model Inference Time')
ax.legend()
plt.show()

print("\n💡 Data has more precision → Posterior closer to sample mean")

---
## Exercise 3: Prior Sensitivity Analysis

**Scenario**: Testing how different prior beliefs affect conclusions.

Data: 8 successes in 10 trials

**Tasks**:
1. Calculate posteriors for 4 different priors
2. Compare posterior means and credible intervals
3. At what sample size does prior choice matter less?

In [ ]:
# Data
successes, trials = 8, 10

# Different priors to test
priors = [
    (1, 1, "Uniform"),
    (0.5, 0.5, "Jeffreys"),
    (5, 5, "Centered at 0.5"),
    (2, 8, "Pessimistic (20%)"),
]

# Your solution here

### Solution

In [ ]:
# Data
successes, trials = 8, 10

# Different priors to test
priors = [
    (1, 1, "Uniform"),
    (0.5, 0.5, "Jeffreys"),
    (5, 5, "Centered at 0.5"),
    (2, 8, "Pessimistic (20%)"),
]

print("PRIOR SENSITIVITY ANALYSIS")
print("="*55)

print(f"\nData: {successes}/{trials} = {successes/trials:.1%}")
print(f"MLE: {successes/trials:.4f}")

# 1 & 2. Calculate posteriors
print(f"\n{'Prior':<20} {'Prior Mean':>12} {'Post Mean':>12} {'95% CI':>25}")
print("-"*75)

theta = np.linspace(0, 1, 200)
fig, ax = plt.subplots(figsize=(12, 6))

results = []
for alpha, beta, name in priors:
    alpha_post = alpha + successes
    beta_post = beta + trials - successes
    
    prior_mean = alpha / (alpha + beta) if (alpha + beta) > 0 else 0.5
    post_mean = alpha_post / (alpha_post + beta_post)
    ci = stats.beta.ppf([0.025, 0.975], alpha_post, beta_post)
    
    results.append((name, post_mean, ci))
    print(f"{name:<20} {prior_mean:>12.4f} {post_mean:>12.4f} [{ci[0]:.4f}, {ci[1]:.4f}]")
    
    ax.plot(theta, stats.beta.pdf(theta, alpha_post, beta_post), lw=2, 
            label=f'{name}: Post={post_mean:.3f}')

ax.axvline(successes/trials, color='black', linestyle='--', alpha=0.5, label=f'MLE={successes/trials:.3f}')
ax.legend()
ax.set_xlabel('θ')
ax.set_ylabel('Density')
ax.set_title(f'Prior Sensitivity with n={trials}')
plt.show()

# 3. At what sample size does prior matter less?
print(f"\n3️⃣ Prior Influence vs Sample Size:")

# Keep same proportion 80% success rate
sample_sizes = [10, 50, 100, 500]

print(f"\n{'n':<8} {'Uniform':>12} {'Pessimistic':>12} {'Difference':>12}")
print("-"*50)

for n in sample_sizes:
    k = int(0.8 * n)
    
    # Uniform prior
    uniform_post = (1 + k) / (2 + n)
    
    # Pessimistic prior
    pess_post = (2 + k) / (10 + n)
    
    diff = abs(uniform_post - pess_post)
    print(f"{n:<8} {uniform_post:>12.4f} {pess_post:>12.4f} {diff:>12.4f}")

print("\n💡 As n increases:")
print("  - Posterior means converge to MLE (0.80)")
print("  - Prior influence diminishes")
print("  - Data 'overwhelms' the prior")
print("\n  Rule of thumb: Prior matters when n < prior 'effective sample size'")
print("  For Beta(α,β): effective n ≈ α + β")

---
## Exercise 4: Posterior Predictive Distribution

**Scenario**: After training, predict model performance on new data.

- Posterior: Beta(25, 10) for accuracy parameter θ
- Predict: accuracy on next batch of 20 samples

**Tasks**:
1. Calculate expected number of correct predictions
2. Find 90% prediction interval
3. Compare with plug-in prediction using posterior mean

In [ ]:
# Posterior
alpha_post, beta_post = 25, 10

# Future prediction
n_future = 20

# Your solution here

### Solution

In [ ]:
# Posterior
alpha_post, beta_post = 25, 10

# Future prediction
n_future = 20

print("POSTERIOR PREDICTIVE DISTRIBUTION")
print("="*55)

theta_mean = alpha_post / (alpha_post + beta_post)
print(f"\nPosterior: Beta({alpha_post}, {beta_post})")
print(f"Posterior mean θ: {theta_mean:.4f}")
print(f"Predicting: {n_future} new samples")

# Calculate Beta-Binomial PMF
k_values = np.arange(0, n_future + 1)
predictive_probs = []

for k in k_values:
    # Beta-Binomial: P(k|n,α,β) = C(n,k) × B(k+α, n-k+β) / B(α, β)
    prob = comb(n_future, k, exact=True) * \
           beta_func(alpha_post + k, beta_post + n_future - k) / \
           beta_func(alpha_post, beta_post)
    predictive_probs.append(prob)

predictive_probs = np.array(predictive_probs)

# 1. Expected number of correct predictions
expected_k = np.sum(k_values * predictive_probs)
print(f"\n1️⃣ Expected correct predictions: {expected_k:.2f}")
print(f"   (Using E[k] = n × E[θ] = {n_future} × {theta_mean:.4f} = {n_future * theta_mean:.2f})")

# 2. 90% Prediction Interval
cumsum = np.cumsum(predictive_probs)
lower_idx = np.searchsorted(cumsum, 0.05)
upper_idx = np.searchsorted(cumsum, 0.95)

print(f"\n2️⃣ 90% Prediction Interval: [{lower_idx}, {upper_idx}]")
print(f"   Probability in interval: {cumsum[upper_idx] - cumsum[lower_idx-1] if lower_idx > 0 else cumsum[upper_idx]:.4f}")

# 3. Compare with plug-in
plugin_probs = stats.binom.pmf(k_values, n_future, theta_mean)

# Variance comparison
var_predictive = np.sum(k_values**2 * predictive_probs) - expected_k**2
var_plugin = n_future * theta_mean * (1 - theta_mean)

print(f"\n3️⃣ Plug-in vs Posterior Predictive:")
print(f"   Plug-in variance:    {var_plugin:.4f}")
print(f"   Predictive variance: {var_predictive:.4f}")
print(f"   Ratio: {var_predictive/var_plugin:.2f}x")

# Plug-in 90% interval
plugin_lower = stats.binom.ppf(0.05, n_future, theta_mean)
plugin_upper = stats.binom.ppf(0.95, n_future, theta_mean)
print(f"   Plug-in 90% interval: [{plugin_lower:.0f}, {plugin_upper:.0f}]")
print(f"   Predictive 90% interval: [{lower_idx}, {upper_idx}]")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PMF comparison
width = 0.35
axes[0].bar(k_values - width/2, predictive_probs, width, label='Posterior Predictive', alpha=0.8)
axes[0].bar(k_values + width/2, plugin_probs, width, label='Plug-in', alpha=0.8)
axes[0].axvline(expected_k, color='red', linestyle='--', label=f'E[k]={expected_k:.1f}')
axes[0].set_xlabel('Number Correct (out of 20)')
axes[0].set_ylabel('Probability')
axes[0].set_title('Predictive Distribution Comparison')
axes[0].legend()

# CDF comparison
axes[1].step(k_values, np.cumsum(predictive_probs), 'b-', lw=2, label='Predictive CDF', where='mid')
axes[1].step(k_values, np.cumsum(plugin_probs), 'g--', lw=2, label='Plug-in CDF', where='mid')
axes[1].axhline(0.05, color='gray', linestyle=':', alpha=0.5)
axes[1].axhline(0.95, color='gray', linestyle=':', alpha=0.5)
axes[1].set_xlabel('Number Correct')
axes[1].set_ylabel('Cumulative Probability')
axes[1].set_title('CDF: Predictive is More Dispersed')
axes[1].legend()

plt.tight_layout()
plt.show()

print("\n💡 Key Insights:")
print("  - Posterior predictive has WIDER variance")
print("  - Accounts for uncertainty in θ, not just sampling variance")
print("  - More honest uncertainty quantification")
print("  - Crucial for calibrated predictions in ML!")